# Module 1 — OCR Văn Bản Chữ Hán Cổ (PaddleOCR-VL-1.6 Trên Google Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caoban123/Tool-Annotation-Sinonom/blob/main/OCR/notebooks/01_OCR_Han.ipynb)

> 🚀 **Chế độ Mặc Định Google Colab (Yêu cầu T4/L4/A100 GPU)**
> - Notebook này thực hiện quy trình OCR chữ Hán chuẩn 12 bước với mô hình Vision-Language **PaddleOCR-VL-1.6**.
> - Khởi tạo GPU: Vào **Runtime** → **Change runtime type** → Chọn **T4 GPU** (hoặc A100/L4).
> - Hệ thống checkpoint từng trang, bảo vệ chống dữ liệu giả (zero-mock policy), tự động đồng bộ UTF-8 và trích xuất `han.txt`.

In [ ]:
# Cell 1: Chẩn đoán môi trường thực thi (Python, OS, PyTorch, CUDA, GPU VRAM)
import os
import sys
import platform

print("=== BÁO CÁO MÔI TRƯỜNG THỰC THI ===")
print(f"Hệ điều hành:     {platform.platform()}")
print(f"Phiên bản Python: {sys.version.split()[0]}")

try:
    import torch
    print(f"PyTorch version:  {torch.__version__}")
    cuda_available = torch.cuda.is_available()
    print(f"CUDA Available:   {cuda_available}")
    if cuda_available:
        device_name = torch.cuda.get_device_name(0)
        total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        bf16_sup = torch.cuda.is_bf16_supported()
        print(f"GPU Thiết bị:     {device_name}")
        print(f"Tổng dung lượng:  {total_mem:.2f} GB VRAM")
        print(f"Hỗ trợ bf16:      {bf16_sup} (Sẽ dùng {'bfloat16' if bf16_sup else 'float16'} cho PaddleOCR-VL)")
    else:
        print("\n⚠️ CẢNH BÁO: Không phát hiện GPU CUDA!")
        print("Mô hình PaddleOCR-VL-1.6 (0.9B VLM) yêu cầu tối thiểu GPU 8GB-16GB VRAM.")
        print("Vui lòng vào menu 'Runtime' -> 'Change runtime type' -> Chọn 'T4 GPU' trước khi tiếp tục.")
except ImportError:
    print("PyTorch chưa được cài đặt. Sẽ được cài đặt ở Cell 2.")


In [ ]:
# Cell 2: Cài đặt thư viện dependencies cần thiết cho PaddleOCR-VL
import sys
import subprocess

print("🚀 Đang cài đặt các thư viện bắt buộc (transformers>=4.49.0, accelerate, pypdfium2...)...\n")
pkgs = [
    "transformers>=4.49.0",
    "accelerate",
    "torch",
    "torchvision",
    "pypdf",
    "pypdfium2",
    "pillow",
    "sentencepiece",
    "einops",
    "gdown",
    "requests",
    "pydantic",
    "matplotlib"
]

cmd = [sys.executable, "-m", "pip", "install", "-q"] + pkgs
subprocess.run(cmd, check=True)
print("✓ Cài đặt hoàn tất dependencies!")


In [ ]:
# Cell 3: Kiểm tra phiên bản thư viện và trạng thái sẵn sàng
import transformers
import accelerate
import torch
import PIL
import pypdfium2

print("=== KIỂM TRA PHIÊN BẢN PACKAGE ===")
print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"torch:        {torch.__version__}")
print(f"pillow:       {PIL.__version__}")
try:
    import importlib.metadata as _im
    _pv = _im.version('pypdfium2')
except Exception:
    _pv = getattr(pypdfium2, '__version__', 'ready')
print(f"pypdfium2:    {_pv}")

assert torch.cuda.is_available(), "LỖI: GPU chưa được kích hoạt. Hãy chuyển Runtime sang GPU T4/A100 trước khi chạy tiếp!"
print("\n✓ Môi trường sẵn sàng cho PaddleOCR-VL!")


In [ ]:
# Cell 4: Đồng bộ mã nguồn từ GitHub & Nạp các module cốt lõi
import os
import sys
import subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_path = Path('/content/Tool-Annotation-Sinonom')
    if not repo_path.exists():
        print("Đang clone repository từ GitHub...")
        subprocess.run(["git", "clone", "https://github.com/caoban123/Tool-Annotation-Sinonom.git", str(repo_path)], check=True)
    else:
        print("Đang cập nhật mã nguồn mới nhất từ main...")
        subprocess.run(["git", "-C", str(repo_path), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", "origin/main"], check=True)
    
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    os.chdir(str(repo_path))
else:
    repo_path = Path('.').resolve()
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

from OCR.utils.ocr_pipeline import PaddleVLOCREngine, OCRPipeline, create_ocr_engine
from OCR.utils.checkpoints import OCRCheckpointManager
from OCR.utils.export_text import export_ocr_result, validate_ocr_output
from OCR.utils.render_pdf import render_pdf_to_images
from OCR.utils.data_loader import inspect_pdf, get_works_catalog, resolve_work, ensure_data_source, download_from_gdrive

print(f"✓ Đã nạp thành công các module từ {repo_path}")


In [ ]:
# Cell 5: Cấu hình tham số OCR tài liệu
# Tùy chọn tác phẩm hoặc tệp PDF:
SELECTED_WORK_KEY = 'DVDDTB'  # DVDDTB, HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, VIET_SU_YEU
START_PAGE = 21               # Trang bắt đầu xử lý (1-based index)
END_PAGE = 30                 # Trang kết thúc xử lý (vd: 30 cho 10 trang, hoặc None cho toàn bộ)
READING_ORDER = 'rtl_vertical'# 'rtl_vertical' (sách Hán cổ đọc từ phải sang trái, từ trên xuống dưới)
ENGINE_NAME = 'PaddleOCR-VL-1.6'
USE_GPU = True
FORCE_REPROCESS = False       # Đặt True nếu muốn OCR lại và ghi đè checkpoint

output_dir = Path('data/ocr_output')
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir = Path('data/ocr_checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("=== CẤU HÌNH THỰC THI OCR ===")
print(f"Tác phẩm:         {SELECTED_WORK_KEY}")
print(f"Trang xử lý:      {START_PAGE} -> {END_PAGE}")
print(f"Thứ tự đọc:       {READING_ORDER}")
print(f"Mô hình OCR:      {ENGINE_NAME}")
print(f"Chế độ GPU:       {USE_GPU}")
print(f"Ghi đè cache:     {FORCE_REPROCESS}")


In [ ]:
# Cell 6: Kiểm tra tệp tin PDF và kết xuất trang mẫu
from PIL import Image
import matplotlib.pyplot as plt

# Tự động kéo toàn bộ 660MB PDF gốc từ Google Drive công khai nếu chưa có trên máy
GDRIVE_FILE_ID = '1buDy5j8bNtFLUzh1yvTSOPX5dAyE_ShG'
ensure_data_source(base_dir='data/source_materials', gdrive_id=GDRIVE_FILE_ID)

catalog = get_works_catalog()
work_info = resolve_work(SELECTED_WORK_KEY, base_dir='data/source_materials')
target_pdf = work_info.get('han_abs_path')

is_scan_image = False
if target_pdf and Path(target_pdf).exists() and target_pdf.endswith('.pdf'):
    pdf_info = inspect_pdf(target_pdf, sample_pages=3)
    total_pdf_pages = pdf_info['total_pages']
    print(f"✓ Tìm thấy PDF: {Path(target_pdf).name}")
    print(f"- Tổng số trang: {total_pdf_pages}")
    print(f"- Định dạng:    {pdf_info['pdf_type']}")
    
    # Điều chỉnh END_PAGE nếu vượt quá tổng số trang
    if END_PAGE is None or END_PAGE > total_pdf_pages:
        END_PAGE = total_pdf_pages
        
    print(f"Đang render trang mẫu {START_PAGE}...")
    sample_images = render_pdf_to_images(target_pdf, start_page=START_PAGE, end_page=START_PAGE, scale=2.0)
    sample_img = sample_images[0]
else:
    print("⚠️ Không tìm thấy PDF trong thư mục source_materials. Sử dụng ảnh scan chuẩn làm mẫu:")
    fixture_path = Path('tests/fixtures/sample_han_scan.png')
    sample_img = Image.open(fixture_path)
    is_scan_image = True
    print(f"✓ Tải ảnh mẫu: {fixture_path}")

print(f"Kích thước ảnh trang mẫu: {sample_img.size} (Width x Height)")
plt.figure(figsize=(8, 10))
plt.imshow(sample_img)
plt.title(f"Trang mẫu OCR (Trang {START_PAGE})")
plt.axis('off')
plt.show()


In [ ]:
# Cell 7: Khởi tạo mô hình Vision-Language PaddleOCR-VL-1.6 thật trên GPU
import time

print(f"🚀 Đang tải mô hình {ENGINE_NAME} từ HuggingFace vào GPU...")
t0 = time.time()

engine = PaddleVLOCREngine(
    model_id="PaddlePaddle/PaddleOCR-VL-1.6",
    use_gpu=USE_GPU
)

load_time = time.time() - t0
print(f"✓ Khởi tạo thành công trong {load_time:.2f} giây!")
print(f"- Hardware device: {engine.hardware}")
print(f"- Is mock:         {engine.is_mock} (Bảo đảm 100% mô hình thật)")

if torch.cuda.is_available() and USE_GPU:
    vram_alloc = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"- VRAM đã dùng:    {vram_alloc:.2f} GB")


In [ ]:
# Cell 8: Kiểm thử cửa ải nhận dạng trang đơn (Gate Test)
# PHẢI vượt qua kiểm thử này trước khi chạy toàn bộ sách!
print(f"🔍 Đang thực hiện nhận dạng AI trang mẫu (Trang {START_PAGE})...")
t_gate_start = time.time()

sample_boxes = engine.recognize_image(sample_img)
t_gate_end = time.time()

# Kiểm tra tính hợp lệ của kết quả OCR
assert isinstance(sample_boxes, list), "Lỗi: Kết quả OCR phải là danh sách bounding box!"
assert len(sample_boxes) > 0, "Lỗi: Không nhận dạng được ký tự nào trên trang mẫu!"

first_text = sample_boxes[0].get('text', '')
assert "cpu_test_adapter" not in getattr(engine, 'engine_type', ''), "Lỗi: Phát hiện mock engine! Phải chạy mô hình thật."

print(f"✓ GATE TEST THÀNH CÔNG trong {t_gate_end - t_gate_start:.2f}s!")
print(f"- Số lượng text boxes: {len(sample_boxes)}")
print(f"- Mẫu văn bản đầu tiên: {first_text}")


In [ ]:
# Cell 9: Đối chiếu trực quan ảnh gốc và văn bản nhận dạng
from OCR.utils.page_order import sort_bounding_boxes

sorted_boxes = sort_bounding_boxes(sample_boxes, reading_order=READING_ORDER)

print(f"=== VĂN BẢN TRANG {START_PAGE} (Thứ tự đọc: {READING_ORDER}) ===")
for idx, box in enumerate(sorted_boxes[:20], 1):
    txt = box.get('text', '').strip()
    conf = box.get('confidence', 1.0)
    print(f"[{idx:02d}] (conf: {conf:.2f}) {txt}")

if len(sorted_boxes) > 20:
    print(f"... (còn {len(sorted_boxes) - 20} dòng văn bản)")


In [ ]:
# Cell 10: Xử lý hàng loạt đa trang với hệ thống Checkpoint chống gián đoạn
chk_mgr = OCRCheckpointManager(cache_dir='data/ocr_checkpoints', allow_mock=False)
pipeline = OCRPipeline(engine=engine, checkpoint_manager=chk_mgr, reading_order=READING_ORDER)

if not is_scan_image and target_pdf and Path(target_pdf).exists():
    print(f"Đang render các trang từ {START_PAGE} đến {END_PAGE}...")
    batch_images = render_pdf_to_images(target_pdf, start_page=START_PAGE, end_page=END_PAGE, scale=2.0)
    source_id = f"{work_info.get('ma_tac_pham', 'doc')}_han"
else:
    print("Đang xử lý ảnh mẫu kiểm thử...")
    batch_images = [sample_img]
    source_id = "sample_han_scan"
    START_PAGE = 1

print(f"🚀 Bắt đầu xử lý {len(batch_images)} trang với PaddleOCR-VL-1.6...")
batch_result = pipeline.process_images(
    images=batch_images,
    source_id=source_id,
    start_page=START_PAGE,
    force_reprocess=FORCE_REPROCESS
)

print("\n✓ Hoàn tất xử lý hàng loạt!")
print(f"- Tổng số trang đã xử lý: {len(batch_result['pages'])}")
for p in batch_result['pages']:
    status_str = "[Từ Cache]" if p.get('is_cached') else f"[AI Mới - {p.get('elapsed_seconds', 0):.2f}s]"
    print(f"  Trang {p['page_number']}: {len(p.get('text', ''))} ký tự {status_str}")


In [ ]:
# Cell 11: Xuất văn bản han.txt và metadata chuẩn UTF-8
out_txt = output_dir / 'han.txt'
out_meta = output_dir / 'han_metadata.json'

# Kiểm tra và xuất file với bảo vệ dữ liệu nghiêm ngặt
export_ocr_result(batch_result, out_txt, out_meta)

print(f"✓ Đã ghi file văn bản Hán: {out_txt}")
print(f"✓ Đã ghi file metadata:    {out_meta}")

print("\n--- TRÍCH ĐOẠN 20 DÒNG ĐẦU CỦA han.txt ---")
with open(out_txt, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if idx >= 20:
            break
        print(line.rstrip())

# Tự động tải về máy nếu đang trên Google Colab
if 'google.colab' in sys.modules:
    from google.colab import files
    print("\nĐang tải file han.txt về máy...")
    files.download(str(out_txt))


In [ ]:
# Cell 12: Kiểm toán chất lượng và Báo cáo thống kê hiệu năng
pages = batch_result.get('pages', [])
total_pages = len(pages)
cached_pages = sum(1 for p in pages if p.get('is_cached'))
new_pages = total_pages - cached_pages
total_chars = sum(len(p.get('text', '')) for p in pages)
avg_chars = total_chars / total_pages if total_pages > 0 else 0
total_inference_time = sum(p.get('elapsed_seconds', 0) for p in pages if not p.get('is_cached'))
avg_time_per_page = total_inference_time / new_pages if new_pages > 0 else 0

print("=" * 50)
print("           BÁO CÁO KIỂM TOÁN CHẤT LƯỢNG OCR")
print("=" * 50)
print(f"Tác phẩm / Nguồn:           {batch_result.get('source_id')}")
print(f"Thang trang:                {batch_result.get('start_page')} -> {batch_result.get('end_page')}")
print(f"Mô hình sử dụng:            {batch_result.get('engine')}")
print(f"Thứ tự đọc:                 {batch_result.get('reading_order')}")
print(f"Tổng số trang xử lý:        {total_pages}")
print(f"- Số trang nạp từ cache:    {cached_pages}")
print(f"- Số trang nhận dạng mới:   {new_pages}")
print(f"Tổng số ký tự Hán trích xuất:{total_chars:,} ký tự")
print(f"Trung bình ký tự / trang:   {avg_chars:.1f} ký tự/trang")
if new_pages > 0:
    print(f"Thời gian suy luận TB:      {avg_time_per_page:.2f} giây/trang")
print("=" * 50)

# Kiểm tra bất thường (trang rỗng hoặc quá ít ký tự)
anomalies = [p for p in pages if len(p.get('text', '').strip()) < 10]
if anomalies:
    print(f"⚠️ Lưu ý: Phát hiện {len(anomalies)} trang có ít hơn 10 ký tự: {[p['page_number'] for p in anomalies]}")
else:
    print("✓ Toàn bộ các trang đều có mật độ văn bản chữ Hán đầy đủ và hợp lệ!")
